In [4]:
import requests
import json

weather_url = "https://api.open-meteo.com/v1/forecast"
weather_params = {
    "latitude": 31.5497,
    "longitude": 74.3436,
    "hourly": "temperature_2m,relative_humidity_2m",
    "timezone": "Asia/Karachi",
}

r = requests.get(weather_url, params=weather_params)
print("Status:", r.status_code)

data = r.json()
print(data.keys())
print(json.dumps(data, indent=2)[:1500])


Status: 200
dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])
{
  "latitude": 31.528997,
  "longitude": 74.38995,
  "generationtime_ms": 0.0591278076171875,
  "utc_offset_seconds": 18000,
  "timezone": "Asia/Karachi",
  "timezone_abbreviation": "GMT+5",
  "elevation": 222.0,
  "hourly_units": {
    "time": "iso8601",
    "temperature_2m": "\u00b0C",
    "relative_humidity_2m": "%"
  },
  "hourly": {
    "time": [
      "2026-11-05T00:00",
      "2026-11-05T01:00",
      "2026-11-05T02:00",
      "2026-11-05T03:00",
      "2026-11-05T04:00",
      "2026-11-05T05:00",
      "2026-11-05T06:00",
      "2026-11-05T07:00",
      "2026-11-05T08:00",
      "2026-11-05T09:00",
      "2026-11-05T10:00",
      "2026-11-05T11:00",
      "2026-11-05T12:00",
      "2026-11-05T13:00",
      "2026-11-05T14:00",
      "2026-11-05T15:00",
      "2026-11-05T16:00",
      "2026-11-05T17:00",
      "20

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

df24 = hist.copy().dropna()

df24["hour"] = df24["time"].dt.hour
df24["aqi_lag1"] = df24["us_aqi"].shift(1)
df24["aqi_lag24"] = df24["us_aqi"].shift(24)   # kal isi waqt ka AQI
df24["target24"] = df24["us_aqi"].shift(-24)   # 24 ghante baad ka AQI

df24 = df24.dropna()

feats = ["hour", "pm2_5", "pm10", "aqi_lag1", "aqi_lag24"]
X = df24[feats]
y = df24["target24"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# baseline: "kal isi waqt jaisa AQI" ya "abhi jaisa"
print("Baseline (abhi jaisa) MAE:", mean_absolute_error(y_test, X_test["us_aqi"] if "us_aqi" in X_test else X_test["aqi_lag1"]))

lr = LinearRegression().fit(X_train, y_train)
print("Linear Regression MAE:", mean_absolute_error(y_test, lr.predict(X_test)))

NameError: name 'hist' is not defined

In [6]:
import requests
import pandas as pd
import sqlite3

url = "https://air-quality-api.open-meteo.com/v1/air-quality"
params = {
    "latitude": 31.5497,
    "longitude": 74.3436,
    "hourly": "pm10,pm2_5,us_aqi",
    "past_days": 30,
    "timezone": "Asia/Karachi",
}
hist = pd.DataFrame(requests.get(url, params=params).json()["hourly"])
hist["time"] = pd.to_datetime(hist["time"])

conn = sqlite3.connect("aqi.db")
hist.to_sql("aqi_data", conn, if_exists="replace", index=False)

print(pd.read_sql("SELECT COUNT(*) AS rows FROM aqi_data", conn))

   rows
0   840


In [7]:
q = """
SELECT DATE(time) AS day, ROUND(AVG(us_aqi),1) AS avg_aqi, MAX(us_aqi) AS max_aqi
FROM aqi_data
GROUP BY DATE(time)
ORDER BY avg_aqi DESC
LIMIT 5
"""
print(pd.read_sql(q, conn))

          day  avg_aqi  max_aqi
0  2026-11-02    205.3      215
1  2026-11-03    202.6      214
2  2026-11-05    199.2      202
3  2026-11-04    195.5      201
4  2026-10-22    191.9      194


In [ ]:
df = hist.copy()

# missing values hatao
df = df.dropna()

# naye columns (features)
df["hour"] = df["time"].dt.hour
df["aqi_lag1"] = df["us_aqi"].shift(1)      # pichle ghante ka AQI
df["pm25_lag1"] = df["pm2_5"].shift(1)      # pichle ghante ka PM2.5
df["target"] = df["us_aqi"].shift(-1)       # agle ghante ka AQI (jo predict karna hai)

df = df.dropna()
print(df.shape)
print(df.head())

In [8]:
q = "SELECT COUNT(*) AS unhealthy_hours FROM aqi_data WHERE us_aqi > 150"
print(pd.read_sql(q, conn))

   unhealthy_hours
0              554


In [ ]:
import joblib
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_mae = mean_absolute_error(y_test, rf.predict(X_test))
lr_mae = mean_absolute_error(y_test, lr.predict(X_test))
print("Linear Regression MAE:", lr_mae)
print("Random Forest MAE:", rf_mae)

best = rf if rf_mae < lr_mae else lr
joblib.dump(best, "aqi_model_24h.pkl")
joblib.dump(feats, "features.pkl")
print("Saved model:", type(best).__name__)

In [9]:
q = """
SELECT strftime('%H', time) AS hour, ROUND(AVG(us_aqi),1) AS avg_aqi
FROM aqi_data
GROUP BY hour
ORDER BY avg_aqi DESC
LIMIT 5
"""
print(pd.read_sql(q, conn))

  hour  avg_aqi
0   18    162.2
1   19    161.1
2   17    160.8
3   20    159.5
4   03    159.0


In [ ]:
print("Baseline (abhi jaisa) MAE:", mean_absolute_error(y_test, X_test["aqi_lag1"]))
print("Baseline (kal isi waqt jaisa) MAE:", mean_absolute_error(y_test, X_test["aqi_lag24"]))

In [ ]:
import requests
import pandas as pd

url = "https://air-quality-api.open-meteo.com/v1/air-quality"
params = {
    "latitude": 31.5497,
    "longitude": 74.3436,
    "hourly": "pm10,pm2_5,us_aqi",
    "past_days": 30,
    "timezone": "Asia/Karachi",
}

r = requests.get(url, params=params)
print("Status:", r.status_code)

hist = pd.DataFrame(r.json()["hourly"])
hist["time"] = pd.to_datetime(hist["time"])
print(hist.shape)
print(hist.describe())


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

features = ["hour", "pm2_5", "pm10", "aqi_lag1", "pm25_lag1"]
X = df[features]
y = df["target"]

# time order mein baanto (shuffle nahi)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)
print("MAE:", mean_absolute_error(y_test, pred))

In [ ]:
baseline_pred = X_test["aqi_lag1"]
print("Baseline MAE:", mean_absolute_error(y_test, baseline_pred))

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print("Random Forest MAE:", mean_absolute_error(y_test, rf_pred))

In [ ]:
import pandas as pd

aq_url = "https://air-quality-api.open-meteo.com/v1/air-quality"
aq_params = {
    "latitude": 31.5497,
    "longitude": 74.3436,
    "hourly": "pm10,pm2_5,us_aqi",
    "timezone": "Asia/Karachi",
}

r = requests.get(aq_url, params=aq_params)
print("Status:", r.status_code)

aq = r.json()
aq_df = pd.DataFrame(aq["hourly"])
aq_df["time"] = pd.to_datetime(aq_df["time"])
print(aq_df.head())
print(aq_df.shape)

In [ ]:
y_train_d = y_train - X_train["aqi_lag1"]

rf2 = RandomForestRegressor(n_estimators=100, random_state=42)
rf2.fit(X_train, y_train_d)

rf2_pred = rf2.predict(X_test) + X_test["aqi_lag1"]
print("RF (change) MAE:", mean_absolute_error(y_test, rf2_pred))

In [ ]:
print(data["hourly"].keys())
print(data["hourly"]["time"][0])
print(data["hourly"]["temperature_2m"][0])


In [ ]:

print(data["hourly"]["time"][1])
print(data["hourly"]["temperature_2m"][1])
print(len(data["hourly"]["time"]))


In [ ]:
import pandas as pd

df = pd.DataFrame(data["hourly"])
print(df.head())
print(df.shape)


In [ ]:
df["time"] = pd.to_datetime(df["time"])
print(df.dtypes)
print(df["temperature_2m"].max(), df["temperature_2m"].min())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(df["time"], df["temperature_2m"])
plt.title("Lahore Temperature - Next 7 Days")
plt.xlabel("Time")
plt.ylabel("Temperature (°C)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(aq_df["time"], aq_df["us_aqi"], color="red")
plt.axhline(150, color="orange", linestyle="--", label="Unhealthy (150)")
plt.title("Lahore Air Quality (US AQI) - Next Days")
plt.xlabel("Time")
plt.ylabel("US AQI")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

print("Max AQI:", aq_df["us_aqi"].max())
print("Average AQI:", aq_df["us_aqi"].mean())